# 00 — Fundamental signals on CPU and GPU

Generate **sine**, **square**, and **sawtooth** waves three ways:

| Stack | Role |
| --- | --- |
| `math` | stdlib only: lists, `math.sin`, `math.floor` |
| SciPy | `numpy` time base + `scipy.signal` waveforms |
| PyTorch | same waveforms as tensors, preferably on **CUDA** |

python-control is the baseline for the `1x` filter notebooks. This notebook only builds the excitation signals those filters will later consume.


In [ ]:
from __future__ import annotations

import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy import signal

%matplotlib inline

print(f"NumPy  {np.__version__}")
print(f"SciPy  {__import__('scipy').__version__}")
print(f"Torch  {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA  {torch.version.cuda}  |  device: {torch.cuda.get_device_name(0)}")
    print(f"capability: {torch.cuda.get_device_capability(0)}")
else:
    print("No GPU in this session — PyTorch cells still run, on CPU.")


In [ ]:
FS_HZ = 8_192          # samples / second
DURATION_S = 0.05      # 50 ms window so individual cycles are visible
F0_HZ = 220.0          # concert A3-ish tone
N = int(FS_HZ * DURATION_S)
DUTY = 0.5             # square-wave duty cycle

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{N} samples @ {FS_HZ} Hz   f0 = {F0_HZ} Hz   device = {device}")


## 1. `math` — no NumPy, no tensors

A time base of Python floats and list comprehensions. Useful as a reference implementation you can read line by line.


In [ ]:
def math_time(n: int, fs: float) -> list[float]:
    return [k / fs for k in range(n)]


def math_sine(t: list[float], f0: float) -> list[float]:
    two_pi = 2.0 * math.pi
    return [math.sin(two_pi * f0 * ti) for ti in t]


def math_square(t: list[float], f0: float, duty: float = 0.5) -> list[float]:
    # Phase in [0, 1): positive for the duty-cycle fraction of each period.
    out = []
    for ti in t:
        phase = (ti * f0) % 1.0
        out.append(1.0 if phase < duty else -1.0)
    return out


def math_sawtooth(t: list[float], f0: float) -> list[float]:
    # Rising ramp in [-1, 1), same convention as scipy.signal.sawtooth.
    out = []
    for ti in t:
        phase = (ti * f0) % 1.0
        out.append(2.0 * phase - 1.0)
    return out


t_math = math_time(N, FS_HZ)
y_sine_math = math_sine(t_math, F0_HZ)
y_square_math = math_square(t_math, F0_HZ, DUTY)
y_saw_math = math_sawtooth(t_math, F0_HZ)

print(
    f"math sine peak={max(y_sine_math):.4f}  "
    f"square unique={sorted(set(y_square_math))}  "
    f"saw range=[{min(y_saw_math):.3f}, {max(y_saw_math):.3f}]"
)


## 2. SciPy — vectorized waveforms

`scipy.signal.square` and `scipy.signal.sawtooth` take a *phase in radians*, so pass `2 * pi * f0 * t`.


In [ ]:
t_np = np.arange(N, dtype=np.float64) / FS_HZ
omega_t = 2.0 * np.pi * F0_HZ * t_np

y_sine_sp = np.sin(omega_t)
y_square_sp = signal.square(omega_t, duty=DUTY)
y_saw_sp = signal.sawtooth(omega_t, width=1.0)

print(y_sine_sp.dtype, y_square_sp[:4], y_saw_sp[:4])


## 3. PyTorch — same formulas, on CUDA when present

Tensors are allocated **already on** `device`. The first CUDA call pays a context-init cost; later calls reuse it.


In [ ]:
def torch_time(n: int, fs: float, device: torch.device) -> torch.Tensor:
    return torch.arange(n, device=device, dtype=torch.float32) / fs


def torch_sine(t: torch.Tensor, f0: float) -> torch.Tensor:
    return torch.sin(2.0 * math.pi * f0 * t)


def torch_square(t: torch.Tensor, f0: float, duty: float = 0.5) -> torch.Tensor:
    phase = torch.remainder(t * f0, 1.0)
    return torch.where(phase < duty, torch.ones_like(t), -torch.ones_like(t))


def torch_sawtooth(t: torch.Tensor, f0: float) -> torch.Tensor:
    phase = torch.remainder(t * f0, 1.0)
    return 2.0 * phase - 1.0


t_th = torch_time(N, FS_HZ, device)
y_sine_th = torch_sine(t_th, F0_HZ)
y_square_th = torch_square(t_th, F0_HZ, DUTY)
y_saw_th = torch_sawtooth(t_th, F0_HZ)

print(y_sine_th.device, y_sine_th.dtype, y_sine_th[:4].tolist())


## Cross-check the three stacks

Max absolute error after bringing the GPU tensors back to host NumPy.


In [ ]:
def max_abs(a, b) -> float:
    return float(np.max(np.abs(np.asarray(a, dtype=np.float64) - np.asarray(b, dtype=np.float64))))


y_sine_gpu = y_sine_th.detach().cpu().numpy()
y_square_gpu = y_square_th.detach().cpu().numpy()
y_saw_gpu = y_saw_th.detach().cpu().numpy()

rows = [
    ("sine", y_sine_math, y_sine_sp, y_sine_gpu),
    ("square", y_square_math, y_square_sp, y_square_gpu),
    ("sawtooth", y_saw_math, y_saw_sp, y_saw_gpu),
]
print(f"{'wave':<10} {'math vs scipy':>14} {'scipy vs torch':>16} {'math vs torch':>14}")
for name, ym, ys, yt in rows:
    print(
        f"{name:<10} {max_abs(ym, ys):14.3e} {max_abs(ys, yt):16.3e} {max_abs(ym, yt):14.3e}"
    )


## Overlay plot


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
waves = [
    ("sine", y_sine_math, y_sine_sp, y_sine_gpu),
    ("square", y_square_math, y_square_sp, y_square_gpu),
    ("sawtooth", y_saw_math, y_saw_sp, y_saw_gpu),
]
ms = np.asarray(t_math) * 1e3
for ax, (name, ym, ys, yt) in zip(axes, waves):
    ax.plot(ms, ym, label="math", lw=2.4, alpha=0.35)
    ax.plot(ms, ys, label="scipy", lw=1.4, ls="--")
    ax.plot(ms, yt, label="torch", lw=1.0, ls=":")
    ax.set_ylabel(name)
    ax.grid(True, alpha=0.3)
axes[0].legend(loc="upper right", ncol=3)
axes[-1].set_xlabel("time (ms)")
axes[0].set_title(f"f0 = {F0_HZ:.0f} Hz   fs = {FS_HZ} Hz   device = {device}")
fig.tight_layout()
plt.show()


## GPU vs CPU timing

Longer buffers so the kernel time dominates Python overhead. `torch.cuda.synchronize()` is required or the timer finishes before the GPU does.


In [ ]:
N_BENCH = 2_000_000
F_BENCH = 1_000.0


def bench_cpu_math(n: int) -> float:
    t = math_time(n, FS_HZ)
    t0 = time.perf_counter()
    math_sine(t, F_BENCH)
    math_square(t, F_BENCH)
    math_sawtooth(t, F_BENCH)
    return time.perf_counter() - t0


def bench_scipy(n: int) -> float:
    t = np.arange(n, dtype=np.float64) / FS_HZ
    w = 2.0 * np.pi * F_BENCH * t
    t0 = time.perf_counter()
    _ = np.sin(w)
    _ = signal.square(w)
    _ = signal.sawtooth(w)
    return time.perf_counter() - t0


def bench_torch(n: int, dev: torch.device) -> float:
    t = torch_time(n, FS_HZ, dev)
    if dev.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    _ = torch_sine(t, F_BENCH)
    _ = torch_square(t, F_BENCH)
    _ = torch_sawtooth(t, F_BENCH)
    if dev.type == "cuda":
        torch.cuda.synchronize()
    return time.perf_counter() - t0


# Warm-up (especially important for CUDA context + kernel compile)
_ = bench_scipy(8_192)
_ = bench_torch(8_192, device)

print(f"{'backend':<16} {'seconds':>10}   n = {N_BENCH:,}")
print(f"{'math (CPU)':<16} {bench_cpu_math(min(N_BENCH, 200_000)):10.4f}   (capped — list loops are slow)")
print(f"{'scipy (CPU)':<16} {bench_scipy(N_BENCH):10.4f}")
print(f"{'torch CPU':<16} {bench_torch(N_BENCH, torch.device('cpu')):10.4f}")
print(f"{'torch ' + str(device):<16} {bench_torch(N_BENCH, device):10.4f}")


## Takeaways

- `math` is the readable ground truth. It will not be the inner loop of a real controller.
- SciPy matches that ground truth to ~1e-15 for these piecewise-linear waves.
- PyTorch float32 on GPU is within ~1e-6 of the double-precision CPU waves — expected, not a bug.
- Next decade (`10`, `11`): put these waves through low-pass and band-pass plants using **python-control**, then re-implement the same filters with `math`, SciPy, and PyTorch.
